<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z332_FeaturesSerieClasica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Features de Serie Clásica — Hurst, Half-Life, ACF, Drawdown

## La idea

En vez de usar el mismo modelo para todos los productos, queremos **elegir el modelo por producto** según las propiedades estadísticas de su serie.

| Feature | Qué mide | Implicación para el modelo |
|---|---|---|
| **Hurst (H)** | Memoria de la serie | H > 0.5 → tiene momentum, la tendencia persiste → reg lineal reciente tiene sentido. H < 0.5 → revierte → naive o mediana |
| **Half-life** | Cuántos meses tarda en volver a la media | Corto → serie revierte rápido → no seguir la tendencia. Largo → la tendencia persiste |
| **ACF lag 1** | Correlación con el mes anterior | Alta → el pasado reciente predice bien → HAR. Baja → ruido |
| **Drawdown máx** | Mayor caída acumulada desde un pico | Alto → producto propenso a quiebres |
| **Sharpe de la serie** | Retorno mensual / desvío | Alto → serie con dirección clara. Bajo → ruido |
| **Z-score reciente** | Cuántas sigmas está el último valor respecto a la historia | Alto negativo → posible quiebre en curso |
| **Market resilience** | Correlación entre shock en t y nivel en t+3 | Alta → se recupera. Baja → el shock persiste |

## Aplicación

Con estas features podemos:
1. **Visualizar** qué tipo de productos tenemos
2. **Elegir el modelo** por producto: si H > 0.5 y half-life largo → reg lineal. Si H < 0.5 → naive/mediana
3. **Usarlas como features** en el panel de regresión (z323) o como static features en AutoGluon (z324)

## 0.1 Init ambiente Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets

descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

descargar  "sell-in.txt.gz"
descargar  "product_id_apredecir201912.txt"

# 1  Setup

In [ ]:
!pip install uv
!uv pip install -q kaggle

In [ ]:
import os
import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.stattools import acf

import warnings
warnings.filterwarnings('ignore')

In [ ]:
PARAM = {
  'experimento': 'FeaturesClasicas-01',
  # umbral Hurst para elegir modelo
  'hurst_momentum': 0.55,   # > esto → reg lineal reciente
  'hurst_reverting': 0.45,  # < esto → naive/mediana
  # ventana para reg lineal cuando H > hurst_momentum
  'ventana_reg': 6,
  'kaggle_competition': 'labo-iii-2026-rosario',
}

In [ ]:
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

# 2  Datos

In [ ]:
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
).sort(["product_id", "periodo"])

tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")
tb_ventas    = tb_ventas.join(tb_apredecir, on="product_id", how="inner").sort(["product_id", "periodo"])

productos = tb_apredecir["product_id"].to_list()
print(f"{len(productos)} productos")

# 3  Cálculo de features clásicas por producto

### Hurst (método R/S simplificado)
```
H = log(R/S) / log(N)
R = max(cumsum de desvíos) - min(cumsum de desvíos)
S = std de la serie
```
H > 0.5 → momentum (la tendencia persiste)  
H = 0.5 → ruido blanco  
H < 0.5 → reversión a la media

### Half-life
Ajusta AR(1): `Δy_t = λ·y_{t-1} + ε`  
Half-life = -log(2) / log(1 + λ)  
Si λ < 0 la serie revierte, half-life mide cuántos períodos tarda en volver a la mitad del desvío.

In [ ]:
def hurst_rs(serie: np.ndarray) -> float:
    """Exponente de Hurst por método R/S."""
    n = len(serie)
    if n < 8:
        return np.nan
    # usamos la serie completa
    mean  = serie.mean()
    desv  = serie - mean
    cumul = np.cumsum(desv)
    R     = cumul.max() - cumul.min()
    S     = serie.std() + 1e-9
    if R <= 0:
        return np.nan
    return float(np.log(R / S) / np.log(n))


def half_life(serie: np.ndarray) -> float:
    """Half-life de reversión a la media via AR(1) en diferencias."""
    if len(serie) < 4:
        return np.nan
    y    = serie[1:]
    y_l  = serie[:-1]
    dy   = y - y_l
    X    = y_l.reshape(-1, 1)
    lam  = float(LinearRegression().fit(X, dy).coef_[0])
    if lam >= 0:
        return np.inf   # no revierte (o tiene momentum)
    return float(-np.log(2) / np.log(1 + lam))


def max_drawdown(serie: np.ndarray) -> float:
    """Máxima caída acumulada desde un pico, normalizada por el pico."""
    if len(serie) < 2:
        return 0.0
    pico    = np.maximum.accumulate(serie)
    drawdown = (pico - serie) / (pico + 1e-9)
    return float(drawdown.max())


def sharpe_serie(serie: np.ndarray) -> float:
    """Retorno mensual promedio / std (como Sharpe de la variación)."""
    if len(serie) < 3:
        return np.nan
    retornos = np.diff(serie) / (serie[:-1] + 1e-9)
    std = retornos.std() + 1e-9
    return float(retornos.mean() / std)


def resilience(serie: np.ndarray, lag: int = 3) -> float:
    """
    Market resilience: correlación entre el shock en t y el nivel relativo en t+lag.
    Shock = retorno en t. Nivel relativo en t+lag = (tn_{t+lag} - tn_t) / tn_t.
    Alta correlación positiva → se recupera. Baja/negativa → el shock persiste.
    """
    if len(serie) < lag + 3:
        return np.nan
    shocks   = np.diff(serie[:-lag]) / (serie[:-lag-1] + 1e-9)
    recovery = (serie[lag+1:] - serie[1:-lag]) / (serie[1:-lag] + 1e-9)
    if len(shocks) < 3 or len(recovery) < 3:
        return np.nan
    n = min(len(shocks), len(recovery))
    corr = float(np.corrcoef(shocks[:n], recovery[:n])[0, 1])
    return corr if np.isfinite(corr) else np.nan


print("Funciones definidas")

In [ ]:
features_list = []

for pid in productos:
    serie = (
        tb_ventas.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )

    # ACF lag 1
    try:
        acf1 = float(acf(serie, nlags=1, fft=False)[1])
    except Exception:
        acf1 = np.nan

    # z-score del último valor respecto a la historia
    media = serie.mean()
    std   = serie.std() + 1e-9
    zscore_ultimo = float((serie[-1] - media) / std)

    # pendiente normalizada últimos 6 meses
    if len(serie) >= 6:
        ult6       = serie[-6:]
        slope_norm = float(np.polyfit(np.arange(6), ult6, 1)[0]) / (media + 1e-9)
    else:
        slope_norm = 0.0

    features_list.append({
        'product_id':      pid,
        'n_meses':         len(serie),
        'media':           float(media),
        'cv':              float(std / (media + 1e-9)),
        'hurst':           hurst_rs(serie),
        'half_life':       half_life(serie),
        'acf_lag1':        acf1,
        'max_drawdown':    max_drawdown(serie),
        'sharpe':          sharpe_serie(serie),
        'resilience_3m':   resilience(serie, lag=3),
        'zscore_ultimo':   zscore_ultimo,
        'pendiente_6m':    slope_norm,
    })

tb_feat = pl.DataFrame(features_list)
print(f"Features calculadas: {tb_feat.shape}")
display(tb_feat.head(10))

# 4  Distribución de features

Entendemos cómo se distribuyen las propiedades estadísticas entre los 780 productos.

In [ ]:
feats_plot = ['hurst', 'half_life', 'acf_lag1', 'max_drawdown', 'sharpe',
              'resilience_3m', 'zscore_ultimo', 'pendiente_6m']

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, feat in enumerate(feats_plot):
    vals = tb_feat[feat].drop_nulls().to_numpy()
    # recortamos outliers extremos para visualizar
    p1, p99 = np.percentile(vals, [1, 99])
    vals_clip = vals[(vals >= p1) & (vals <= p99)]

    axes[i].hist(vals_clip, bins=40, color='steelblue', edgecolor='white', linewidth=0.4)
    axes[i].set_title(feat, fontsize=9)
    axes[i].axvline(np.median(vals_clip), color='red', linestyle='--', linewidth=0.8,
                    label=f'mediana={np.median(vals_clip):.2f}')
    axes[i].legend(fontsize=6)

    if feat == 'hurst':
        axes[i].axvline(0.5, color='orange', linewidth=1.2, label='H=0.5')

fig.suptitle('Distribución de features estadísticas — 780 productos', fontsize=11)
plt.tight_layout()
plt.show()

# resumen del Hurst
h = tb_feat['hurst'].drop_nulls().to_numpy()
print(f"Hurst mediana: {np.median(h):.3f}")
print(f"  H > 0.55 (momentum):  {(h > 0.55).sum()} productos ({(h > 0.55).mean()*100:.1f}%)")
print(f"  H 0.45-0.55 (neutro): {((h >= 0.45) & (h <= 0.55)).sum()} productos")
print(f"  H < 0.45 (reversión): {(h < 0.45).sum()} productos ({(h < 0.45).mean()*100:.1f}%)")

# 5  Hurst vs Half-life — el mapa de regímenes

Cada punto es un producto. Este es el mapa que nos dice qué modelo usar por producto.

- **Arriba-derecha** (H alto, half-life largo): momentum fuerte, la tendencia persiste → reg lineal reciente
- **Abajo-izquierda** (H bajo, half-life corto): reversión rápida → naive o mediana
- **Zona media**: neutros, cualquier modelo es similar

In [ ]:
df_plot = tb_feat.to_pandas()
# half-life infinito (momentum puro) lo capeamos para visualizar
df_plot['half_life_cap'] = df_plot['half_life'].clip(upper=60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hurst vs half-life
sc = axes[0].scatter(
    df_plot['hurst'], df_plot['half_life_cap'],
    c=df_plot['pendiente_6m'], cmap='RdYlGn',
    s=20, alpha=0.7, vmin=-0.3, vmax=0.3
)
plt.colorbar(sc, ax=axes[0], label='pendiente 6m normalizada')
axes[0].axvline(0.5, color='black', linestyle='--', linewidth=0.8, label='H=0.5')
axes[0].axvline(PARAM['hurst_momentum'],  color='green', linestyle=':', linewidth=1, label=f'H>{PARAM["hurst_momentum"]} → reg lineal')
axes[0].axvline(PARAM['hurst_reverting'], color='red',   linestyle=':', linewidth=1, label=f'H<{PARAM["hurst_reverting"]} → naive')
axes[0].set_xlabel('Hurst', fontsize=9)
axes[0].set_ylabel('Half-life (meses, cap=60)', fontsize=9)
axes[0].set_title('Mapa de regímenes\n(color = tendencia reciente: rojo=caída, verde=subida)', fontsize=9)
axes[0].legend(fontsize=7)

# ACF lag 1 vs z-score último
axes[1].scatter(
    df_plot['acf_lag1'], df_plot['zscore_ultimo'],
    c=df_plot['max_drawdown'], cmap='Reds',
    s=20, alpha=0.7
)
plt.colorbar(sc, ax=axes[1], label='max drawdown')
axes[1].axhline(-2, color='red', linestyle='--', linewidth=0.8, label='z=-2 (caída fuerte)')
axes[1].axhline( 2, color='green', linestyle='--', linewidth=0.8, label='z=+2 (subida fuerte)')
axes[1].axvline(0, color='black', linestyle='--', linewidth=0.5)
axes[1].set_xlabel('ACF lag 1', fontsize=9)
axes[1].set_ylabel('Z-score último mes', fontsize=9)
axes[1].set_title('Memoria vs estado actual\n(color = drawdown máximo histórico)', fontsize=9)
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

# 6  Modelo adaptativo — elige según Hurst y half-life

Regla simple:
- H > `hurst_momentum` **y** half-life > 6 meses → **reg lineal reciente** (la tendencia persiste)
- H < `hurst_reverting` **o** half-life < 3 meses → **mediana 6m** (la serie revierte)
- Resto → **HAR** (estructura mixta)

In [ ]:
def pred_reg_lineal(serie, ventana=6, horizonte=2):
    w = min(ventana, len(serie))
    if w < 2:
        return max(float(serie.mean()), 0.0)
    y = serie[-w:]
    x = np.arange(w).reshape(-1, 1)
    pred = float(LinearRegression().fit(x, y).predict([[w - 1 + horizonte]])[0])
    return max(pred, 0.0)

def pred_har(serie, horizonte=2):
    if len(serie) < 14:
        return max(float(serie[-12:].mean()), 0.0)
    T = len(serie)
    X, y = [], []
    for t in range(12, T):
        X.append([serie[t-1], serie[t-3:t].mean(), serie[t-6:t].mean(), serie[t-12:t].mean()])
        y.append(serie[t])
    m = LinearRegression().fit(np.array(X), np.array(y))
    def next_pred(s):
        t = len(s)
        return float(m.predict([[s[t-1], s[t-3:t].mean(), s[t-6:t].mean(), s[t-12:t].mean()]])[0])
    p1 = max(next_pred(serie), 0.0)
    if horizonte == 2:
        return max(next_pred(np.append(serie, p1)), 0.0)
    return p1


feat_dict = {row['product_id']: row for row in tb_feat.iter_rows(named=True)}

resultados = []
conteo_modelo = {'reg_lineal': 0, 'mediana': 0, 'har': 0}

for pid in productos:
    serie = (
        tb_ventas.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )
    f  = feat_dict[pid]
    H  = f['hurst']    if f['hurst']    is not None else 0.5
    hl = f['half_life'] if f['half_life'] is not None else 6.0
    if np.isinf(hl): hl = 60.0

    if H > PARAM['hurst_momentum'] and hl > 6:
        pred   = pred_reg_lineal(serie, ventana=PARAM['ventana_reg'], horizonte=2)
        modelo = 'reg_lineal'
    elif H < PARAM['hurst_reverting'] or hl < 3:
        pred   = max(float(np.median(serie[-6:])), 0.0)
        modelo = 'mediana'
    else:
        pred   = pred_har(serie, horizonte=2)
        modelo = 'har'

    conteo_modelo[modelo] += 1
    resultados.append({'product_id': pid, 'tn': pred, 'modelo': modelo})

tb_adapt = pl.DataFrame(resultados)

print("Distribución de modelos asignados:")
for m, n in conteo_modelo.items():
    print(f"  {m:12s}: {n:4d} productos ({n/len(productos)*100:.1f}%)")

# 7  Visualización de ejemplos por régimen

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 9))

for row_i, modelo_nombre in enumerate(['reg_lineal', 'har', 'mediana']):
    pids_m = (
        tb_adapt.filter(pl.col('modelo') == modelo_nombre)
        ['product_id'].head(3).to_list()
    )
    for col_i, pid in enumerate(pids_m):
        serie = (
            tb_ventas.filter(pl.col("product_id") == pid)
            .sort("periodo")["tn"].to_numpy().astype(float)
        )
        f     = feat_dict[pid]
        pred  = float(tb_adapt.filter(pl.col('product_id') == pid)['tn'][0])

        ax = axes[row_i][col_i]
        ax.plot(range(len(serie)), serie, 'o-', color='steelblue', markersize=3, linewidth=1.3)
        ax.scatter([len(serie) + 1], [pred], color='tomato', s=80, zorder=5,
                   marker='D', label=f'pred={pred:.1f}')
        ax.set_title(
            f'pid {pid}  [{modelo_nombre}]\nH={f["hurst"]:.2f}  hl={f["half_life"]:.1f}m  acf1={f["acf_lag1"]:.2f}',
            fontsize=7
        )
        ax.legend(fontsize=6)

fig.suptitle('Ejemplos por régimen asignado', fontsize=10)
plt.tight_layout()
plt.show()

# 8  Guardar features y submit

In [ ]:
# guardar features para usar en AutoGluon (static features) o panel (z323/z324)
tb_feat.write_csv('features_series_clasicas.csv')
import shutil
shutil.copy('features_series_clasicas.csv',
            '/content/.drive/My Drive/labo3/exp/features_series_clasicas.csv')
print("Features guardadas")

In [ ]:
def kaggle_submit(competencia, archivo, mensaje):
    import os
    os.system(f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"')

tb_final = tb_adapt.select(['product_id', 'tn'])
tb_final.write_csv('AdaptativoHurst.csv')
kaggle_submit(
    PARAM['kaggle_competition'],
    'AdaptativoHurst.csv',
    f'Modelo adaptativo: reg{PARAM["ventana_reg"]}m/HAR/mediana segun Hurst y half-life'
)

# 9  Qué probar

| Cambio | Dónde | Por qué |
|---|---|---|
| `hurst_momentum: 0.60` | PARAM | Más exigente para usar reg lineal — solo los de momentum fuerte |
| `ventana_reg: 3` | PARAM | Más reactivo en los de momentum |
| Agregar ACF como criterio | Sección 6 | Si acf_lag1 > 0.5 → HAR, si < 0 → mediana |
| Usar zscore_ultimo como criterio | Sección 6 | Si z < -2 → reg lineal (ya en caída), si z > 2 → mediana (subida probablemente temporal) |
| Pasar tb_feat como static_features a AutoGluon | z324 | Los modelos profundos (TFT, DeepAR) pueden explotar Hurst, half-life, etc. |
| Usar half-life como peso en ensemble | post-submit | Ponderar reg lineal vs naive según cuánto revierte la serie |